In [0]:
import importlib
from config.config import *

from delta.tables import DeltaTable
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
def inspect_silver_columns(path):

    entries = dbutils.fs.ls(path)

    for entry in entries:

        if entry.isDir():

            table_name = entry.name.rstrip("/")

            try:
                df = spark.read.format("delta").load(entry.path)

                print("\n" + "=" * 80)
                print(f"SILVER TABLE: {table_name}")
                print("=" * 80)

                for field in df.schema.fields:
                    print(f"{field.name:<40} {field.dataType}")

            except Exception as e:
                print(f"Could not inspect {table_name}: {e}")


inspect_silver_columns(silver_path)

In [0]:
tables = [
    "estat_tour_occ_arm",
    "estat_tour_occ_mnor",
    "estat_tour_occ_nim",
    "un_tourism_accommodation_hotels",
    "un_tourism_inbound_arrivals",
    "world_bank_international_arrivals"
]

for table in tables:
    df = spark.read.format("delta").load(f"{silver_path}/{table}")

    print(f"\n{'=' * 70}")
    print(f"TABLE: {table}")
    print(f"{'=' * 70}")

    # Row count
    print("Total rows:", df.count())

    # Null counts
    null_counts = df.select([
        count(
            when(col(c).isNull(), c)
        ).alias(c)
        for c in df.columns
    ])

    print("Null counts:")
    null_counts.show(truncate=False)

    # Numeric columns: check negative values
    numeric_cols = [
        field.name for field in df.schema.fields
        if isinstance(field.dataType, (DoubleType, IntegerType))
    ]

    for col_name in numeric_cols:
        negative_count = df.filter(
            col(col_name) < 0
        ).count()

        if negative_count > 0:
            print(f"Negative values in {col_name}: {negative_count}")


In [0]:
for table, metric in [
    ("estat_tour_occ_arm", "tourism_value"),
    ("estat_tour_occ_nim", "nights_spent")
]:
    df = spark.read.format("delta").load(
        f"{silver_path}/{table}"
    )

    print(f"\n{'=' * 70}")
    print(f"TABLE: {table}")
    print(f"{'=' * 70}")

    df.filter(col(metric) < 0) \
        .groupBy("unit_code", "observation_flag") \
        .count() \
        .orderBy(desc("count")) \
        .show(50, truncate=False)

In [0]:
for table, metric in [
    ("estat_tour_occ_arm", "tourism_value"),
    ("estat_tour_occ_mnor", "occupancy_rate"),
    ("estat_tour_occ_nim", "nights_spent")
]:
    df = spark.read.format("delta").load(
        f"{silver_path}/{table}"
    )

    print(f"\n{'=' * 70}")
    print(f"TABLE: {table}")
    print(f"{'=' * 70}")

    df.filter(col(metric).isNull()) \
        .groupBy("unit_code", "observation_flag") \
        .count() \
        .orderBy(desc("count")) \
        .show(30, truncate=False)

In [0]:
for table, metric in [
    ("estat_tour_occ_arm", "tourism_value"),
    ("estat_tour_occ_mnor", "occupancy_rate"),
    ("estat_tour_occ_nim", "nights_spent")
]:
    df = spark.read.format("delta").load(
        f"{silver_path}/{table}"
    )

    print(f"\n{table}")

    df.filter(col(metric).isNull()) \
        .groupBy(year("period").alias("year")) \
        .count() \
        .orderBy("year") \
        .show(30, truncate=False)

In [0]:
for table, metric in [
    ("estat_tour_occ_arm", "tourism_value"),
    ("estat_tour_occ_nim", "nights_spent")
]:
    df = spark.read.format("delta").load(
        f"{silver_path}/{table}"
    )

    print(f"\n{table}")

    df.groupBy("unit_code") \
        .agg(
            count("*").alias("total_rows"),
            count(metric).alias("available_values"),
            min("period").alias("first_period"),
            max("period").alias("last_period")
        ) \
        .show(truncate=False)

In [0]:
hotels = spark.read.format("delta").load(
    f"{silver_path}/un_tourism_accommodation_hotels"
)

hotels.filter(
    col("indicator_code").contains("figures are reflective")
).show(truncate=False)

# UN Hotels: inspect unknown geography
hotels.filter(
    col("geography_type") == "unknown"
).select(
    "reporter_area",
    "reporter_area_code",
    "indicator_code",
    "indicator_label",
    "unit",
    "reporter_country_iso3"
).distinct().show(100, truncate=False)


# UN Hotels: inspect null units
hotels.filter(
    col("unit").isNull()
).select(
    "reporter_area",
    "indicator_code",
    "indicator_label"
).distinct().show(100, truncate=False)


# World Bank: inspect records without ISO3
wb = spark.read.format("delta").load(
    f"{silver_path}/world_bank_international_arrivals"
)

wb.filter(
    col("country_iso3").isNull()
).select(
    "country_name",
    "country_code",
    "geography_type"
).distinct().orderBy("geography_type", "country_name").show(
    300, truncate=False
)

In [0]:
hotels = spark.read.format("delta").load(
    f"{silver_path}/un_tourism_accommodation_hotels"
)

hotels_clean = hotels.filter(
    col("reporter_area_code").isNotNull() &
    col("year").isNotNull() &
    col("hotel_capacity").isNotNull()
)

hotels_clean.write.format("delta").mode("overwrite").save(
    f"{silver_path}/un_tourism_accommodation_hotels"
)

In [0]:
from pyspark.sql.functions import col

# World Bank
wb = spark.read.format("delta").load(
    f"{silver_path}/world_bank_international_arrivals"
)

print("WORLD BANK")
wb.filter(col("country_iso3").isNull()) \
    .select("country_name", "country_code", "geography_type") \
    .distinct() \
    .orderBy("country_code") \
    .show(300, truncate=False)


# UN inbound arrivals
un_arrivals = spark.read.format("delta").load(
    f"{silver_path}/un_tourism_inbound_arrivals"
)

print("UN INBOUND ARRIVALS - REPORTER")
un_arrivals.filter(col("reporter_country_iso3").isNull()) \
    .select("reporter_area_code", "reporter_area", "reporter_geography_type") \
    .distinct() \
    .orderBy("reporter_area_code") \
    .show(300, truncate=False)


print("UN INBOUND ARRIVALS - PARTNER")
un_arrivals.filter(col("partner_country_iso3").isNull()) \
    .select("partner_area_code", "partner_area", "partner_geography_type") \
    .distinct() \
    .orderBy("partner_area_code") \
    .show(300, truncate=False)


# UN accommodation hotels
hotels = spark.read.format("delta").load(
    f"{silver_path}/un_tourism_accommodation_hotels"
)

print("UN ACCOMMODATION HOTELS")
hotels.filter(col("reporter_country_iso3").isNull()) \
    .select("reporter_area_code", "reporter_area", "geography_type") \
    .distinct() \
    .orderBy("reporter_area_code") \
    .show(300, truncate=False)


# Eurostat tables
for table in [
    "estat_tour_occ_arm",
    "estat_tour_occ_mnor",
    "estat_tour_occ_nim"
]:
    df = spark.read.format("delta").load(f"{silver_path}/{table}")

    print(f"EUROSTAT - {table}")
    df.filter(col("country_iso3").isNull()) \
        .select("geo_code", "geography_type") \
        .distinct() \
        .orderBy("geo_code") \
        .show(100, truncate=False)